# Component 5 — Data Modelling & Model Evaluation
## End-to-End Hourly Traffic Volume Forecasting

This notebook trains, evaluates, validates, and refines regression models using the prepared traffic dataset.

**Models included**
1. Linear Regression
2. Random Forest Regressor
3. Gradient Boosting Regressor

The final notebook also includes:
- chronological train/test split
- MAE, RMSE and R²
- cross-validation
- actual vs predicted plot
- residual diagnostics
- feature importance
- model comparison
- hyperparameter refinement


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV

import joblib


## 2. Load the Prepared Dataset

The notebook first tries to use `traffic_preprocessed.csv`. If it is not available, it rebuilds the required temporal features directly from `integrated_traffic_dataset.csv`.


In [ ]:
try:
    df = pd.read_csv("traffic_preprocessed.csv")
except FileNotFoundError:
    df = pd.read_csv("integrated_traffic_dataset.csv")
    df["DateTime"] = pd.to_datetime(
        df["DateTime"], format="mixed", dayfirst=True, errors="coerce"
    )
    df = df.drop_duplicates().reset_index(drop=True)
    df["Hour"] = df["DateTime"].dt.hour
    df["DayOfWeek"] = df["DateTime"].dt.dayofweek
    df["Month"] = df["DateTime"].dt.month
    df["Is_Weekend"] = df["weekend"].astype(int)
    df = df.dropna(subset=["DateTime", "Vehicles"]).reset_index(drop=True)

print("Dataset Shape:", df.shape)
display(df.head())


## 3. Define Features and Target

`Vehicles` is the prediction target. Junction, temporal features and weather measurements are used as predictors.


In [ ]:
features = [
    "Junction",
    "Hour",
    "DayOfWeek",
    "Month",
    "Is_Weekend",
    "temperature_2m (°C)",
    "relative_humidity_2m (%)",
    "precipitation (mm)",
    "wind_speed_10m (km/h)"
]

target = "Vehicles"

X = df[features].copy()
y = df[target].astype(float)

print("Features:", features)
print("Target:", target)


## 4. Chronological Train/Test Split

Because this is a forecasting problem, the final validation set is kept chronologically after the training data. This avoids using future records to train the model for the final holdout evaluation.


In [ ]:
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


## 5. Preprocessing Pipeline

Junction is categorical when applicable, while the remaining predictors are numerical. The pipeline handles missing values and one-hot encodes categorical data.


In [ ]:
categorical_features = []
if X["Junction"].dtype == "object":
    categorical_features = ["Junction"]

numerical_features = [
    c for c in features if c not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numerical_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ],
    remainder="drop"
)

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)


# Model Development and Training

## 6. Train Three Regression Models

We use three models to establish a baseline and compare a linear approach with non-linear ensemble methods.


In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=150,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor),
        ("model", GradientBoostingRegressor(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=5,
            random_state=42
        ))
    ])
}

predictions = {}
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)


## 7. Model Comparison Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(results_df["Model"], results_df["R2"])
axes[0].set_title("R² Comparison")
axes[0].set_ylabel("R²")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(results_df["Model"], results_df["RMSE"])
axes[1].set_title("RMSE Comparison")
axes[1].set_ylabel("RMSE")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


# Model Evaluation & Cross Validation

## 8. Random Forest Evaluation

Random Forest is inspected in more detail because it can capture non-linear relationships between traffic, time, junction and weather.


In [ ]:
rf_model = models["Random Forest"]
rf_pred = predictions["Random Forest"]

mae = mean_absolute_error(y_test, rf_pred)
rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
r2 = r2_score(y_test, rf_pred)

print("Random Forest Test Metrics")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


## 9. Actual vs Predicted Traffic


In [ ]:
n = min(150, len(y_test))

plt.figure(figsize=(14, 5))
plt.plot(
    y_test.iloc[:n].values,
    label="Actual Traffic",
    linewidth=2
)
plt.plot(
    rf_pred[:n],
    label="Random Forest Prediction",
    linestyle="--",
    linewidth=2
)
plt.title("Actual vs Predicted Traffic Volume — First 150 Test Records")
plt.xlabel("Test Time Step")
plt.ylabel("Vehicles")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Residual Analysis

Residuals are calculated as:

**Residual = Actual − Predicted**

The residual distribution and residual-vs-predicted plot help identify systematic error patterns.


In [ ]:
residuals = y_test.values - rf_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=40)
axes[0].set_title("Residual Distribution")
axes[0].set_xlabel("Residual")
axes[0].set_ylabel("Frequency")

axes[1].scatter(rf_pred, residuals, alpha=0.25, s=8)
axes[1].axhline(0, linestyle="--")
axes[1].set_title("Residuals vs Predicted")
axes[1].set_xlabel("Predicted Vehicles")
axes[1].set_ylabel("Residual")

plt.tight_layout()
plt.show()


## 11. Five-Fold Cross Validation

Cross-validation is performed on the training data to measure how consistently the model performs across different subsets. The final chronological test set remains untouched for the final holdout evaluation.


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=kf,
    scoring="r2",
    n_jobs=1
)

print("Cross-validation R² scores:", np.round(cv_scores, 4))
print("Mean CV R²:", round(cv_scores.mean(), 4))
print("Std CV R²:", round(cv_scores.std(), 4))


# Model Refinement

## 12. Hyperparameter Tuning

A compact GridSearchCV is used to refine the Random Forest model. The search is deliberately limited so that it remains practical on the supplied dataset.


In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_grid = {
    "model__n_estimators": [100, 150],
    "model__max_depth": [None, 15],
    "model__min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)


## 13. Evaluate the Refined Model


In [ ]:
refined_model = grid_search.best_estimator_
refined_pred = refined_model.predict(X_test)

refined_mae = mean_absolute_error(y_test, refined_pred)
refined_rmse = np.sqrt(mean_squared_error(y_test, refined_pred))
refined_r2 = r2_score(y_test, refined_pred)

refined_results = pd.DataFrame([{
    "Model": "Refined Random Forest",
    "MAE": refined_mae,
    "RMSE": refined_rmse,
    "R2": refined_r2
}])

display(refined_results)


## 14. Refined Model — Actual vs Predicted


In [ ]:
n = min(150, len(y_test))

plt.figure(figsize=(14, 5))
plt.plot(y_test.iloc[:n].values, label="Actual Traffic", linewidth=2)
plt.plot(
    refined_pred[:n],
    label="Refined Random Forest",
    linestyle="--",
    linewidth=2
)
plt.title("Refined Random Forest — Actual vs Predicted")
plt.xlabel("Test Time Step")
plt.ylabel("Vehicles")
plt.legend()
plt.tight_layout()
plt.show()


## 15. Feature Importance

Feature importance is extracted from the fitted Random Forest after preprocessing. For a categorical Junction column, one-hot encoded features are shown separately.


In [ ]:
best_preprocessor = refined_model.named_steps["preprocessor"]
best_rf = refined_model.named_steps["model"]

feature_names = best_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": best_rf.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance_df.head(15))

top = importance_df.head(15).sort_values("Importance")

plt.figure(figsize=(10, 6))
plt.barh(top["Feature"], top["Importance"])
plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 16. Save Model Outputs

The trained refined model and model-comparison table are saved so they can be reused later.


In [ ]:
joblib.dump(
    refined_model,
    "traffic_volume_random_forest_refined.pkl"
)

results_df.to_csv(
    "model_comparison_results.csv",
    index=False
)

refined_results.to_csv(
    "refined_model_results.csv",
    index=False
)

print("Saved:")
print("- traffic_volume_random_forest_refined.pkl")
print("- model_comparison_results.csv")
print("- refined_model_results.csv")


## 17. Final Project Summary

The completed modelling workflow includes:

- data loading and preparation
- chronological train/test split
- preprocessing pipeline
- multiple regression models
- MAE, RMSE and R² evaluation
- model comparison
- cross-validation
- residual diagnostics
- hyperparameter refinement
- feature importance
- saving the final model

**Important:** Run all cells in order after placing `integrated_traffic_dataset.csv` in the same folder as this notebook.
